# Phase 3: Fine-tuning on a free Colab T4

**Hinglish sentiment: fine-tuned small models vs a large LLM.** This notebook runs the same code as the local scripts (`scripts/finetune_sweep.py`).

| Option | Model | What is trained | Runs | Time on T4 |
|---|---|---|---|---|
| A: encoder | XLM-R base, MuRIL base (raw and emoji→text) | all weights | 9 | ~40 min |
| B: decoder | Qwen3-1.7B, 4-bit (QLoRA) | LoRA adapters + head (~1% of weights) | 4 | ~1 h |

**Before you start**
1. `Runtime → Change runtime type → T4 GPU`.
2. Add your W&B key as a Colab secret: the 🔑 icon in the left sidebar → `Add new secret` → name `WANDB_API_KEY`, value = your key from https://wandb.ai/authorize → toggle **Notebook access** on.
3. `GITHUB_REPO` in the next cell already points to this repo (change it if you fork).

Everything (models, results) is written to **Google Drive**, so if Colab disconnects, re-run all cells; finished runs are skipped.

In [ ]:
GITHUB_REPO = "https://github.com/anonymouse-19/hinglish-sentiment-finetune-vs-llm.git"
DRIVE_DIR = "/content/drive/MyDrive/hinglish-ft"   # models + reports live here

## 1. GPU, Drive, code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull --ff-only
else:
    !git clone {GITHUB_REPO} /content/repo
%cd /content/repo
!git log -1 --format="code version: %h %s"

## 2. Install (Colab's own torch is kept)

In [ ]:
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__, "CUDA:", torch.cuda.is_available())

## 3. Experiment tracking (W&B)

In [ ]:
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
import wandb
wandb.login()

## 4. Rebuild the data (~1 min)
The data is never committed (no licence, D-004); this rebuilds the identical splits from the pinned source.

In [ ]:
!python scripts/prepare_data.py

## 5. Pipeline check on tiny models (~1 min)
Catches environment problems before spending GPU time.

In [ ]:
!python scripts/finetune_sweep.py --family encoder --smoke --only xlmr_lr2e-05

## 6. Option A: encoder sweep (9 runs)
Each run: up to 4 epochs, early stopping on val macro-F1. Only the best-so-far model is kept on Drive.
Watch live curves in W&B: project **hinglish-sentiment-finetune**.

In [ ]:
!python scripts/finetune_sweep.py --family encoder --output-root {DRIVE_DIR}/outputs --report-dir {DRIVE_DIR}/reports

## 7. Option B: Qwen3-1.7B QLoRA sweep (4 runs)
If you run out of GPU time, it resumes from the last finished run next session.

In [ ]:
!python scripts/finetune_sweep.py --family decoder --output-root {DRIVE_DIR}/outputs --report-dir {DRIVE_DIR}/reports

## 8. Bring the results home
Downloads `finetune_reports.zip` (metrics, predictions, sweep tables; no model weights, no tweet text).
Unzip it into the project folder on your laptop so that you get `reports/finetune/encoder/...` and `reports/finetune/decoder/...`.

In [ ]:
!rm -rf /content/bundle /content/finetune_reports.zip && mkdir -p /content/bundle/reports/finetune
!cp -r {DRIVE_DIR}/reports/* /content/bundle/reports/finetune/ && cd /content/bundle && zip -qr /content/finetune_reports.zip reports && unzip -l /content/finetune_reports.zip | tail -3
from google.colab import files
files.download("/content/finetune_reports.zip")